## Make a static maps

### Download libraries and dataset

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

from matplotlib.colors import Normalize

#######################################################

filename = '../data/cleaned/clean_dataset.csv'

#shp_name = '../shapefiles/Belgium-4-Digit-Postcodes-2020.shp'
shp_name = '../shapefiles/belgium_map_simplified.shp'

results_dir = '../map_visualization_results/'

#######################################################

df = pd.read_csv(filename,delimiter=',')

col = 'url'  # Drop the column Url because it needn't for analysis
if col in df.columns:
    df.drop(columns=[col], inplace=True)

## Create cdata for maps

In [2]:
df_avg = df.groupby('postal_code', as_index=False)['price'].median()
df_avg.rename(columns={'price': 'median_price'}, inplace=True)

df_count = df.groupby('postal_code').size().reset_index(name='count')

df_high = df[df['price'] > 500_000]
df_high_count = df_high.groupby('postal_code').size().reset_index(name='high_price_count')

df_summary = pd.merge(df_avg, df_count, on='postal_code', how='left')
df_summary = df_summary.merge(df_high_count, on='postal_code', how='left')

df_summary['high_price_count'] = df_summary['high_price_count'].fillna(0).astype(int)

###########################################################

df['price_per_m2'] = df['price'] / df['area']

df_price_per_m2 = df.groupby('postal_code', as_index=False)['price_per_m2'].median()
df_price_per_m2.rename(columns={'price_per_m2': 'median_price_per_m2'}, inplace=True)

df_summary = df_summary.merge(df_price_per_m2, on='postal_code', how='left')


## prepare shapefile, join it with data

In [3]:
gdf = gpd.read_file(shp_name)

gdf['nouveau_PO'] = pd.to_numeric(gdf['nouveau_PO'], errors='coerce').astype('Int64')
df_summary['postal_code'] = pd.to_numeric(df_avg['postal_code'], errors='coerce').astype('Int64')

gdf = gdf.merge(df_summary, left_on='nouveau_PO', right_on='postal_code', how='left')

## Load some specific libraries

In [4]:
import folium
import geopandas as gpd
import branca.colormap as cm
from folium.features import GeoJsonTooltip

### Create interactive map

In [5]:
# Create folium map
fmap = folium.Map(location=[50.85, 4.35], zoom_start=8, tiles=None)

# Create scale
'''
colormap = cm.linear.Reds_09.scale(
    gdf['median_price_per_m2'].min(),
    gdf['median_price_per_m2'].max()
)'''
colormap = cm.linear.Reds_09.scale(500,5000)

colormap.caption = 'Median Price per m² (€)'

# 3. Add layer
folium.GeoJson(
    gdf,
    style_function=lambda feature: {
        'fillColor': colormap(feature['properties']['median_price_per_m2'])
        if feature['properties']['median_price_per_m2'] is not None else 'transparent',
        'color': 'black',
        'weight': 0.5,
        'fillOpacity': 0.7
    },
    tooltip=GeoJsonTooltip(
        fields=['postal_code', 'median_price_per_m2'],
        aliases=['Postal code:', 'Median price per m² (€):'],
        localize=True,
        sticky=True
    )
).add_to(fmap)

# Add legend
colormap.add_to(fmap)

#######################################################
# Save the map into HTML file

outname = 'interactive_map_price_per_m2.html'

filepath = os.path.join(results_dir, outname)

#fmap.save(filepath)

# Show the map in JupNotebook
fmap